# SIT225 Data Capture Technologies
## Credit Task 5C — Live smooth Plotly Dash update for smartphone accelerometer data

**Name:** _[your name]_ &nbsp;&nbsp;|&nbsp;&nbsp; **Student ID:** _[your ID]_ &nbsp;&nbsp;|&nbsp;&nbsp;
**Date:** _[submission date]_

---

### Contents

| | |
|---|---|
| **Q1** | Method for smooth graph updates |
| **Q2** | Documentation of the `smooth_graph()` wrapper function |
| **Q3** | Video demonstration |
| **Q4** | GitHub repository |
| | Acknowledgements and appendix |

---
# Q1. Method for smooth graph updates

## 1.1 The problem

In the Week 8 activity the Dash graph was updated by waiting for a fixed number of new samples
(N = 20), then building a completely new figure and returning it from the callback. Watching it
run, three separate things make it uncomfortable to look at:

| # | What happens | What the user sees |
|---|---|---|
| 1 | The callback waits for 20 new samples before doing anything | The line stands still, then lurches sideways by 20 points at once |
| 2 | A new `go.Figure` is built and returned every update | The chart is torn down and re-rendered, so it flickers |
| 3 | The axes are auto-scaled and the figure object is replaced | The axis ranges snap to new values, and any zoom the user applied is discarded |

Fixing only one of these is not enough. Removing the batching still leaves the flicker; removing
the flicker still leaves the axes jumping. All three had to be addressed.

## 1.2 Overview of the method

The design separates the program into two halves that run on different clocks and never block
each other. Data arriving from the phone is written into a list. A timer, running independently,
takes whatever is in that list and appends it to the lines that are already drawn.

![Flow diagram of the smooth update method](flow_diagram.png)

The list in the middle is the part that matters. Before, the drawing step was triggered *by* the
data arriving, so the display inherited the phone's timing, which is slow and uneven. Now the
drawing step runs on a fixed 100 ms timer and simply takes whatever is available. If nothing has
arrived, the callback raises `PreventUpdate` and nothing is sent.

## 1.3 The three mechanisms

| Mechanism | Implementation | Problem it solves |
|---|---|---|
| **Separate the two clocks** | Incoming samples are appended to a Python list guarded by a `threading.Lock`. A `dcc.Interval` component drains that list every 100 ms. | Problem 1. The graph no longer waits for a batch of N samples. Most ticks add one or two points, which is small enough that the eye reads it as movement rather than a jump. |
| **Append instead of rebuild** | The callback writes to the graph's `extendData` property instead of its `figure` property. Dash passes this to `Plotly.extendTraces`, which adds points to the existing traces in the browser. | Problem 2. The figure object is never replaced, so there is no teardown and no flicker. It also sends far less data: a few numbers per tick instead of the entire series. |
| **Control the axes explicitly** | `uirevision="keep"` is set in the layout, and a second callback uses `Patch()` to set `xaxis.range` to a window that slides forward each tick. | Problem 3. `uirevision` tells Plotly to preserve zoom and pan across updates. Setting the range explicitly means it moves by a small fixed amount instead of snapping to a recalculated auto-range. |

### Why the two callbacks are kept separate

`extendData` and `figure` are two different properties of the same `dcc.Graph`. They are driven by
two separate callbacks on purpose. Returning a complete figure from one callback would replace the
data that the other callback had just appended. A `Patch()` only carries the instruction *"set
`layout.xaxis.range` to these two numbers"*, so it composes safely with the appended points.

### What this does not do

Arduino Cloud delivers at roughly 2 Hz while the chart refreshes at 10 Hz, so between two
arriving samples there is genuinely nothing new to draw and the newest point holds its position
for a few ticks. The window keeps scrolling throughout, so there is no jump and no redraw, but the
right-hand edge of the line advances in steps rather than continuously.

Interpolating between arriving samples would smooth that out. It was left out on purpose: the
points it would draw are estimates of readings that were never taken, and the three mechanisms
above already remove the batching, the flicker and the axis snapping that the task asks about.

## 1.4 Evidence — before and after

> **[TO DO: run both versions, screenshot each, and save the images next to this notebook as
> `before.png` and `after.png`. Then write two or three sentences under each saying what you
> actually saw — that is the part a screenshot cannot show.]**

**Before — batched redraw**

![Before](before.png)

_[Your observation: describe the jump, the flicker, and what happened when you tried to zoom while
it was updating.]_

**After — smooth update**

![After](after.png)

_[Your observation: describe how the line moves now, and confirm that zooming stays put.]_

## 1.5 Code — the "before" version

Kept for comparison. The three problems from section 1.1 are marked in the comments.

In [ ]:
# BEFORE — the Week 8 behaviour (not run here; see the working notebook)

before_app = Dash(__name__)
before_app.layout = html.Div([
    dcc.Graph(id="g"),
    dcc.Interval(id="timer", interval=500),
])
drawn = [0]


@before_app.callback(Output("g", "figure"), Input("timer", "n_intervals"))
def redraw(_):
    if len(before_data) - drawn[0] < 20:        # problem 1: waits for a batch
        raise PreventUpdate
    drawn[0] = len(before_data)

    t0 = before_data[0][0]
    fig = go.Figure()                            # problem 2: a brand new figure
    for name in ["x", "y", "z"]:
        fig.add_trace(go.Scatter(
            x=[p[0] - t0 for p in before_data],
            y=[p[1][name] for p in before_data],
            mode="lines", name=name))
    # problem 3: no uirevision, no fixed range
    fig.update_layout(xaxis_title="Seconds", yaxis_title="g")
    return fig

## 1.6 Code — the "after" version

The complete smooth application. Everything from sections 1.2 and 1.3 sits inside `smooth_graph()`,
so the application itself is four lines.

In [ ]:
# AFTER — the smooth version

from dash import Dash
from smooth_dash import smooth_graph

app = Dash(__name__)
layout, add_point = smooth_graph(app, ["x", "y", "z"], y_range=(-2, 2))
app.layout = layout

# ...then, from the Arduino Cloud callbacks:
#     add_point({"x": 0.12, "y": -0.31, "z": -0.98})

### Connecting it to the phone

The three axes arrive from Arduino Cloud as three independent variables with three independent
callbacks, so they have to be recombined into one sample before being sent to the graph.
`got_value()` holds each value until all three have arrived, then sends them together and clears.

All three callbacks are invoked from the single `ArduinoCloudClient` thread, so they cannot run at
the same time as each other and no lock is needed at this point. The lock inside `smooth_graph()`
is still required, because that list is read by Dash on a different thread.

In [ ]:
latest = {}


def got_value(axis, value):
    """x, y and z arrive separately. Collect all three, then send one point."""
    latest[axis] = value
    if len(latest) == 3:
        add_point(dict(latest))
        latest.clear()


client = ArduinoCloudClient(
    device_id=os.environ["ARDUINO_DEVICE_ID"],
    username=os.environ["ARDUINO_DEVICE_ID"],
    password=os.environ["ARDUINO_SECRET_KEY"],
    sync_mode=False,
)
client.register("accelerometer_x", value=None, on_write=lambda c, v: got_value("x", v))
client.register("accelerometer_y", value=None, on_write=lambda c, v: got_value("y", v))
client.register("accelerometer_z", value=None, on_write=lambda c, v: got_value("z", v))
threading.Thread(target=client.start, daemon=True).start()

---
# Q2. Documentation — the `smooth_graph()` wrapper function

## 2.1 Purpose

`smooth_graph()` gives a Dash application a live line chart that updates smoothly, for any source
of continuous data. The caller supplies values as they arrive and does not have to know about
`extendData`, `Patch`, `uirevision`, background threads or locks.

It is written for the situation where data arrives irregularly, or more slowly than a chart should
refresh — sensor readings, network measurements, queue depths, prices. Sources faster than the
display rate also work, because points are buffered and drawn in batches.

**Module:** `smooth_dash.py` &nbsp;&nbsp;|&nbsp;&nbsp; **Requires:** `dash >= 2.11`, `plotly`

## 2.2 Usage

Two lines to set up, one line each time data arrives.

```python
from dash import Dash
from smooth_dash import smooth_graph

app = Dash(__name__)
layout, add_point = smooth_graph(app, ["x", "y", "z"], y_range=(-2, 2))
app.layout = layout

add_point({"x": 0.1, "y": -0.3, "z": -0.98})   # call whenever data arrives

app.run()
```

`add_point` is safe to call from any thread, so it can be dropped straight into an existing
callback, socket handler or polling loop without restructuring anything around it.

## 2.3 Parameters

```python
smooth_graph(app, names, window=15, fps=10, y_range=None, title="")
```

| Parameter | Type | Default | Description |
|---|---|---|---|
| `app` | `Dash` | required | The application the callbacks are registered on. |
| `names` | `list[str]` | required | One name per line on the chart. The keys of the dict passed to `add_point` must match these names. |
| `window` | `int` or `float` | `15` | Width of the visible time window, in seconds. The newest point sits at the right-hand edge. |
| `fps` | `int` | `10` | How many times a second the chart updates. |
| `y_range` | `(low, high)` or `None` | `None` | Fixed y-axis range. `None` lets Plotly scale the axis, which is less stable to watch. |
| `title` | `str` | `""` | Chart title. |

## 2.4 Return value

A tuple of two items:

| Returned | Type | Use |
|---|---|---|
| `layout` | `html.Div` | Assign to `app.layout`, or nest inside a larger layout. Contains the `dcc.Graph` and the `dcc.Interval`. |
| `add_point` | function | Call as `add_point(values)` where `values` is a dict keyed by `names`. Thread-safe. Returns nothing. |

**Raises:** `KeyError` from `add_point` if the dict is missing one of the names given in `names`.

## 2.5 Source

In [ ]:
# smooth_dash.py

import time
import threading

import plotly.graph_objects as go
from dash import dcc, html, Input, Output, Patch
from dash.exceptions import PreventUpdate


def smooth_graph(app, names, window=15, fps=10, y_range=None, title=""):
    """Add a smoothly-updating live graph to a Dash app. Returns (layout, add_point)."""
    waiting = []                  # points that have arrived but are not drawn yet
    lock = threading.Lock()       # data arrives on another thread, so we need this
    start = time.time()

    # ---- 1. the user calls this when new data arrives ----------------------
    def add_point(values):
        with lock:
            waiting.append((time.time() - start, values))

    # ---- 2. an empty graph with one line per name --------------------------
    figure = go.Figure()
    for name in names:
        figure.add_trace(go.Scatter(x=[], y=[], mode="lines", name=name))
    figure.update_layout(
        title=title,
        uirevision="keep",            # <- keeps the user's zoom when we update
        xaxis=dict(range=[0, window], title="Seconds"),
        yaxis=dict(range=list(y_range) if y_range else None),
        height=400,
    )

    layout = html.Div([
        dcc.Graph(id="smooth-graph", figure=figure),
        dcc.Interval(id="smooth-timer", interval=int(1000 / fps)),
    ])

    # ---- 3. add the waiting points to the lines ----------------------------
    # "extendData" tells Plotly to ADD to the lines already on screen, instead
    # of drawing a new graph. This is the bit that stops the flicker.
    @app.callback(Output("smooth-graph", "extendData"),
                  Input("smooth-timer", "n_intervals"))
    def draw_new_points(_):
        with lock:
            batch = list(waiting)
            waiting.clear()

        if not batch:
            raise PreventUpdate       # nothing new, do nothing

        times = [point[0] for point in batch]
        lines = []
        for name in names:
            lines.append([point[1][name] for point in batch])

        return (
            dict(x=[times] * len(names), y=lines),
            list(range(len(names))),  # which lines to add to
            1000,                     # keep the newest 1000 points
        )

    # ---- 4. slide the time window so the graph scrolls ---------------------
    # Patch() changes ONE thing (the x axis range) instead of the whole graph.
    @app.callback(Output("smooth-graph", "figure"),
                  Input("smooth-timer", "n_intervals"))
    def slide_window(_):
        left = max(0, (time.time() - start) - window)
        patch = Patch()
        patch["layout"]["xaxis"]["range"] = [left, left + window]
        return patch

    return layout, add_point

## 2.6 Example — a completely different data source

The function should work for data that has nothing to do with an accelerometer. Below it is driven
by two made-up signals at a different rate, with a different axis range and a different number of
lines. `smooth_dash.py` is not modified in any way.

In [ ]:
other_app = Dash(__name__)
layout, add_other = smooth_graph(other_app, ["temperature", "humidity"],
                                 window=20, y_range=(0, 100),
                                 title="Same function, different data")
other_app.layout = layout


def fake_sensor():
    temp, hum = 22.0, 55.0
    while True:
        temp = min(35, max(15, temp + random.gauss(0, 0.4)))
        hum = min(90, max(30, hum + random.gauss(0, 1.0)))
        add_other({"temperature": temp, "humidity": hum})
        time.sleep(0.5)


threading.Thread(target=fake_sensor, daemon=True).start()

> **[TO DO: screenshot this running and save it as `generic.png`.]**

![Generic demo](generic.png)

## 2.7 Design decisions

> **[TO DO: these are the reasons behind the API as it stands. Read them, decide whether you agree,
> and rewrite them in your own words. Your tutor can ask you about any of these in
> Discuss/Demonstrate, so the wording should be yours.]**

**Returning a function rather than asking for one.** `smooth_graph()` hands back `add_point` and
the caller decides when to call it. The alternative would be to accept a `get_data()` function and
poll it. Returning a function means the source stays in control, so the same wrapper works for a
callback-driven source such as Arduino Cloud, a blocking read loop, or a socket handler, without
the wrapper needing to know which.

**A dict rather than positional arguments.** `add_point({"x": .., "y": .., "z": ..})` is longer to
type than `add_point(x, y, z)`, but it cannot be silently wrong. Passing y and z the wrong way
round in a positional call produces a chart that looks fine and is incorrect. With a dict, a wrong
or missing key raises `KeyError` immediately.

**Nothing is invented.** Only points that were actually received are drawn. Interpolating between
samples would make the line glide, but those points would be estimates presented as readings.

**Two callbacks rather than one.** Explained in section 1.3. This is a constraint of how Dash
applies updates, not a stylistic preference.

## 2.8 Limitations

| Limitation | Reason | Workaround |
|---|---|---|
| One graph per application | The component ids `smooth-graph` and `smooth-timer` are fixed strings, so a second call to `smooth_graph()` on the same app would register duplicate ids. | Add an `id_prefix` argument and build the ids from it. |
| Time must move forwards | Points are appended in the order they arrive. A sample with an earlier timestamp will draw a line backwards. | Not handled; sources with out-of-order delivery would need a sort before the buffer. |
| The newest point holds still between arrivals | The source is slower than the refresh rate, and no points are invented. | Interpolate between samples, accepting that those points are estimates. |
| Fixed cap of 1000 points per line | Keeps browser memory bounded, but is not adjustable. | Could be exposed as a parameter. |
| The y-axis does not adapt | `y_range` is fixed or fully automatic, with nothing in between. | Fixed range is recommended for sensor data with a known scale. |

---
# Q3. Video demonstration

> **[TO DO: record the video, upload to Panopto via CloudDeakin, check the share permission, and
> paste the link below.]**

**Video link:** _[paste your Panopto link here]_

**What the video shows:**

| Time | Content |
|---|---|
| 0:00 | Phone streaming into the Arduino IoT Cloud dashboard |
| 0:00 | The "before" version running — the batched redraw |
| 0:00 | The "after" version running — the smooth update |
| 0:00 | Moving the phone and seeing the graph respond live |
| 0:00 | Walkthrough of `smooth_dash.py` |
| 0:00 | The same function driven by unrelated data |

_[Fill in the timestamps once it is recorded.]_

---
# Q4. GitHub repository

> **[TO DO: push the folder, screenshot the GitHub page showing the week-8.2C folder contents, save
> it as `github.png`, and paste the repository link below. Add your tutor as a collaborator.]**

**Repository:** _[paste your repository URL here]_

**Path:** `SIT225_2024T2/week-8.2C/`

**Tutor added as collaborator:** _[yes / date added]_

![GitHub folder contents](github.png)

| File | Contents |
|---|---|
| `smooth_dash.py` | The `smooth_graph()` wrapper function |
| `dash_app.py` | The accelerometer dashboard using the wrapper |
| `SIT225-5C.ipynb` | Working notebook — builds the solution step by step |
| `accelerometer_5c.csv` | Recorded accelerometer data |
| `flow_diagram.png` | The diagram in section 1.2 |
| `screenshots/` | Graph screenshots used in this report |
| `README.md` | How to install and run |
| `.gitignore` | Excludes credentials and cache files |

Arduino Cloud credentials are read from environment variables (`ARDUINO_DEVICE_ID` and
`ARDUINO_SECRET_KEY`) and are not committed to the repository.

---
# Acknowledgements

> **[TO DO: this is the template from the task brief. Fill in the tools you actually used and what
> you used them for. If you used anything besides what is listed, add it.]**

This assessment work was developed with assistance from the following approved Deakin genAI tools:

* **Claude (accessed September 2026):** Used to _[describe what you asked for — for example,
  explaining how Plotly's `extendData` property works, drafting the structure of the wrapper
  function, and reviewing the interpolation logic]_.

All genAI-generated suggestions were critically evaluated and integrated only where supported by
evidence. The final design, decisions, and justifications are my own.

---

## Appendix A — genAI prompts and raw responses

> **[TO DO: paste the prompts you used and the raw responses. The brief requires these in full.]**

## Appendix B — changelog

> **[TO DO: list what you changed from the genAI output and why. For example: which parameters you
> adjusted after testing, anything you removed as unnecessary, anything you rewrote after checking
> it against the Dash documentation, and the results of your own testing.]**

| # | Suggestion received | What I changed | Why |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |